# 2HRX9P6HKXA8V

In [ ]:
# Imports
from foodcast.imports import *
os.chdir(find_project_root())
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, _, _, _, DATA_DIR_3_7 = DATA_DIR_3_x

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = '2HRX9P6HKXA8V'
df_uncleaned = load_single_restaurant(loc_id)
df_targeted = pd.read_parquet(DATA_DIR_3_7 / f'{loc_id}.parquet')

In [ ]:
# Remapping was manual, but stored in a yaml now to stay organized
labeling_path = Path('scripts') / 'labeling'
remapping_path = labeling_path / 'remapping' / 'loc2_remappings.yaml'
with open(remapping_path, "r", encoding="utf-8") as f:
    remapping = yaml.load(f, Loader=yaml.FullLoader)

# Calculate rare items dynamically (items that appear less than 10 times)
rare = df_uncleaned['item_name'].value_counts().to_frame('count').query('count < 10').index.tolist()

# Extract the parts of the yaml to do the programmatic relabeling
df_relabeled = fully_relabel_and_consolidate(
    df                        = df_uncleaned,
    name_changes              = remapping.get("name_changes", {}),
    modification_name_changes = remapping.get("modification_name_changes", []),
    vegan_list                = remapping.get("vegan_list", []),
    vegetarian_list           = remapping.get("vegetarian_list", []),
    meat_list                 = remapping.get("meat_list", []),
    alcohol_list              = remapping.get("alcoholic_drinks", []),
    drinks_list               = remapping.get("non_alcoholic_drinks", []),
    merch                     = remapping.get("merch_list", []),
    rare                      = rare + remapping.get("rare_list", []),
    unknown                   = remapping.get("unknown_list", []),
    remove_categories         = ["Merch", "Drink", "Rare", "Alcohol"]
)

# # Are the labels uniquely specified?
# display(df_relabeled.groupby('item_name')['vegan'].unique()) 

df_relabeled.to_parquet(DATA_DIR_3_1 / (loc_id + '_sales_and_menu.parquet'))

# Consolidate dishes
df_consolidated = rename_items(
    df = df_relabeled,
    name_changes = {
        "Beyond Sausage": ["Meat Beyond Sausage","Vegetarian Beyond Sausage"],
        "Veggie Wurst": ["Meat Veggie Wurst", "Vegan Veggie Wurst"],
        "Vegan Chili": ["Vegetarian Vegan Chili"],
        "Potato Chips": ["Vegetarian Potato Chips"],
        "Veggie Soup": ["Meat Veggie Soup", "Vegetarian Veggie Soup"],
        "Vegetarian": ["Meat Vegetarian", "Vegan Vegetarian"]
    }
)

# # There should be multiple labels for each item
# display(df_consolidated.groupby('item_name')['vegan'].unique()) 

df_consolidated.to_parquet(DATA_DIR_3_2 / (loc_id + '_sales_and_menu.parquet'))

to_dish_time_series(df_consolidated).to_csv(labeling_path / 'timelines' / (loc_id + '.csv'))

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_min=7.75, legend_max=18.70, shift_adjustment=1/15)

In [ ]:
items_less_than_2_dollars = (
    df_targeted
    .groupby('item_name')
    ['unit_price']
    .max()
    .to_frame(name='max_unit_price')
    .query('max_unit_price < 200.0')
    .reset_index()
    #.query('~item_name.isin(["Pumpkin Bread","Muffin Marionberry Cobbler","Muffin Chocolate"])')
    .item_name
    .tolist()
    )
missed_merch = ['Add Ons','Gluten Free Bun']
missed_drinks = []
take_and_go = []

modification_name_changes = [
    
]

name_changes = {
    'Chicken Sausage': ['Organic Chicken Sausage'],
    'Organic Chicken':['Chicken'],
    'Currywurst':['Curt\'S Currywurst'],
    'Chili Con Carne':['Chili'],
    'Bavarian Cream Of Potato Soup':['Soup'],
    'German Potato Salad':['G.P.S.']
}

filtered = (
    df_targeted
    .query('~item_name.isin(@items_less_than_2_dollars)')
    .query('~item_name.isin(@missed_merch)')
    .query('~item_name.isin(@missed_drinks)')
    .query('~item_name.isin(@take_and_go)')
    .query('~item_name.isin(@cookies)')
    .pipe(lambda df: rename_items(df, name_changes))
    #.pipe(lambda df: rename_items_by_modifications(df, modification_name_changes))
)

print(
    filtered
    .groupby('item_name')
    ['item_modifications']
    .apply(lambda s: s.value_counts().index.str.slice(0,20).tolist()[0:3])
    .loc[filtered.item_name.value_counts().index.tolist()]
    .to_frame()
    .join(filtered.item_name.value_counts())
    .reset_index()
    .set_index('count')
    [['item_name','item_modifications']]
    #.iloc[10:,:]
    .to_string()
)

plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

In [ ]:
presence_dict = {}
for item, group in filtered.groupby("item_name"):
    presence_dict[item] = infer_active_days(group["created_at"], max_gap_days=120)
presence_df = pd.concat(presence_dict, axis=1).fillna(False)

presence_weekly = presence_df.resample('W').max().fillna(False).astype(bool).to_period('W')
dish_order = list(filtered.item_name.value_counts().index)
plot_boolean_time_series(presence_weekly, loc_id, before_after_details_true, dish_order, [])
plot_dish_time_series(filtered.set_index('created_at'), loc_id, before_after_details_true, top_n=70)

presence_daily = strict_bridge_fill(presence_df, limit=7).resample('D').max()
#presence_daily = (presence_weekly.astype(float).replace(0.0, np.nan)[vegetarian_dishes].resample('D').interpolate(limit=6).fillna(0).astype(int).sum(axis=1).plot())
menu = pd.read_csv(Path("scripts") / "labeling" / "dish_labels" / (loc_id + ".csv"))

vegan_dishes = menu.loc[menu["vegan"], "item_name"]
vegetarian_dishes = menu.loc[menu["vegetarian"], "item_name"]
mpbamod_dishes = menu.loc[menu["mpbamod"], "item_name"]

vegan_dishes_count = presence_daily[vegan_dishes].sum(axis=1)
vegetarian_dishes_count = presence_daily[vegetarian_dishes].sum(axis=1)
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
indicator = pd.to_datetime(promo_date) < presence_daily.index
mpbamod_dishes_count = presence_daily[mpbamod_dishes].sum(axis=1) * indicator
menu_counts = pd.concat([vegan_dishes_count, vegetarian_dishes_count, mpbamod_dishes_count], axis=1).set_axis(['vegan_dishes_count', 'vegetarian_dishes_count', 'mpbamod_dishes_count'], axis=1)
menu_counts.plot()
menu_counts.to_csv(Path("scripts") / "labeling" / "dish_counts" / (loc_id + ".csv"))

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Beyond")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Beyond")')['item_quantity'].sum())

plot_time_series_subset(
    df_uncleaned.query('item_name.str.contains("Beyond")'), 
    exposure=promo_date, 
    freq='W', 
    truncate=False)
plt.show()

In [ ]:
print(df_consolidated
      .query('~vegetarian')
      ['unit_price']
      .mean())
print((df_consolidated
       .query('~vegetarian')
       ['item_name']
       .nunique()) / 
      (df_consolidated
       ['item_name']
       .nunique()))
plt.plot(df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         (df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique()) / 
         (df_consolidated
         .query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique()), 
         'o', 
         alpha=0.5)
plt.show()

In [ ]:
import matplotlib.dates as mdates



# Define ordering and unmatched items outside the function
ordering = [
    'Big Bob Bratwurst', 'Warm Bavarian Pretzel', 'Hans’ Jalapeño & Cheddar',
    'Helga’s Giant Kelbassi', 'Big City Beef Frank', 'German Potato Salad',
    'Tim’s Cascade Chips', 'Veggie Wurst', 'Dirtyface Beer Wurst',
    'Organic Turkey Dog', 'Oma’s Weisswurst', 'Beyond Sausage', 'Potato Soup',
    'Chili con Carne', 'Bockwurst', 'Vegan Lentil Soup', 'Vegan Chili',
    'Organic Chicken & Apple Sausage', 'Spinach Organic Chicken Sausage',
    'Italian Organic Chicken Sausage', 'Mediterranean Chicken Sausage',
    'Curt’s Currywurst', 'Beyond Sausage Intro', 'COVID Closure',
    'COVID Limited', 'Currywurst Removed', 'Holiday Special',
    'New Chicken Sausage Intro', 'Oktoberfest', 'Weisswurst Intro'
]

unmatched = [
    'Beyond Sausage Intro', 'COVID Closure', 'COVID Limited',
    'Currywurst Removed', 'Holiday Special', 'New Chicken Sausage Intro',
    'Oktoberfest', 'Weisswurst Intro'
]

    
deepresearch = (pd.read_csv('data/Muchen Haus Timeline.csv', index_col=0)
                .pipe(lambda x: x.set_index(pd.to_datetime(x.index).tz_localize('UTC')).to_period('W')))

plot_boolean_time_series(deepresearch, loc_id, before_after_details_true, ordering, unmatched)
plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_min=7.75, legend_max=18.70, shift_adjustment=1/15)

In [ ]:
# display(time_differences_details[loc_id])
# display(time_differences[loc_id])

# # # 10 Hour Difference
# # df['item_name'].value_counts().sort_values(ascending=False).head(10)
# # df.loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

# # # 20 Hour Difference
# # df.loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)
# # time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]